# Frog Species Spectrogram Viewer

Same idea as `Frog_Genus_Display_Function.ipynb`, but narrower: the species
stays put and only the spectrogram rolls. Useful once you already know which
species you want to look at.

**Run `Processing_Frog_Spectrograms_Multiple_Bursts.ipynb` first.**

In [ ]:
import random
import re
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PIL import Image

DISCRETE_SIGNALS_DIR = Path.home() / "Discrete_Signals"
FROG_SPECS_DIR = DISCRETE_SIGNALS_DIR / "Cropped_Frogs_Specs"   # flat, not per-species
FROG_IMAGE_GLOB = "{genus}_{species}_*_spectrogram.png"
FROG_ID_PATTERN = r"_(\d+)_spectrogram\.[^.]+$"   # File_ID is the clip's audio_num


def find_frog_spectrograms(genus, species):
    """Sorted list of spectrogram image paths for a frog species."""
    pattern = FROG_IMAGE_GLOB.format(genus=genus, species=species)
    return sorted(FROG_SPECS_DIR.glob(pattern))


def frog_rows_for_image(df, genus, species, image_path):
    """Every row matching image_path's File_ID (one per burst type detected in
    that clip); falls back to the species' first row."""
    species_rows = df[(df["Genus"] == genus) & (df["Species"] == species)]
    if species_rows.empty:
        return species_rows
    match = re.search(FROG_ID_PATTERN, image_path.name)
    file_id = match.group(1) if match else None
    hit = species_rows[species_rows["File_ID"].astype(str) == str(file_id)]
    if not hit.empty:
        return hit
    return species_rows.iloc[[0]]


def print_frog_burst_rows(rows):
    """Print each burst type's stats for a set of rows sharing one spectrogram."""
    if "Burst_Pattern" in rows.columns:
        pattern = rows.iloc[0]["Burst_Pattern"]
        n_types = rows.iloc[0]["N_Burst_Types_Detected"]
        print(f"Burst pattern: {pattern} ({n_types} burst type(s) detected)")
    print()
    for _, row in rows.iterrows():
        label = f"Burst type {row['Burst_Type']}" if "Burst_Type" in rows.columns else "Stats"
        print(f"  {label}:")
        print(f"    Element length:         {row['Element_Length']}")
        print(f"    Inter-element interval: {row['Inter-Element_Interval']}")
        print(f"    Inter-burst interval:   {row['Inter-Burst_Interval']}")
        print(f"    Elements per burst:     {row['Elements_Per_Burst']} "
              f"(min {row['Min_Elements_Per_Burst']}, max {row['Max_Elements_Per_Burst']})")
        if "N_Bursts_This_Type" in rows.columns:
            print(f"    N bursts this type:     {row['N_Bursts_This_Type']}")
        print()


def render_spectrogram(image_path):
    try:
        image = Image.open(image_path)
    except Exception as e:
        print(f"Could not load image: {e}")
        return
    figure, axes = plt.subplots(figsize=(12, 4))
    axes.imshow(image, cmap="gray")
    axes.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(figure)

In [ ]:
%store -r frog_final_multi

In [ ]:
def ordered_unique_species(df):
    """List of (genus, species) pairs in first-appearance order."""
    pairs = []
    seen = set()
    for genus, species in zip(df["Genus"], df["Species"]):
        if (genus, species) not in seen:
            seen.add((genus, species))
            pairs.append((genus, species))
    return pairs


class FrogSpeciesViewer:
    """Show one randomly chosen spectrogram for the current (fixed) species."""

    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.species_list = ordered_unique_species(self.df)
        self.species_index = 0
        self.current_image_path = None

        self.output = widgets.Output()
        self.prev_species = widgets.Button(description="← Previous species", layout=widgets.Layout(width="180px"))
        self.next_species = widgets.Button(description="Next species →", layout=widgets.Layout(width="180px"))
        self.randomize = widgets.Button(description="New random spectrogram", layout=widgets.Layout(width="200px"))
        self.prev_species.on_click(lambda b: self.change_species(-1))
        self.next_species.on_click(lambda b: self.change_species(1))
        self.randomize.on_click(lambda b: self.pick_random_and_show())

        display(widgets.HBox([self.prev_species, self.next_species]))
        display(self.randomize)
        display(self.output)
        self.pick_random_and_show()

    def change_species(self, direction):
        self.species_index = min(max(self.species_index + direction, 0), len(self.species_list) - 1)
        self.pick_random_and_show()

    def pick_random_and_show(self):
        genus, species = self.species_list[self.species_index]
        images = find_frog_spectrograms(genus, species)
        self.current_image_path = random.choice(images) if images else None
        self.show()

    def show(self):
        with self.output:
            clear_output(wait=True)
            genus, species = self.species_list[self.species_index]
            n_species = len(self.species_list)

            if self.current_image_path is None:
                print(f"Species {self.species_index + 1}/{n_species}: {genus} {species}")
                print("\nNo spectrogram found.")
                return

            rows = frog_rows_for_image(self.df, genus, species, self.current_image_path)
            if rows.empty:
                print(f"No data found for {genus} {species}.")
                return

            print(f"Species {self.species_index + 1}/{n_species}: {genus} {species}")
            print_frog_burst_rows(rows)
            render_spectrogram(self.current_image_path)

In [ ]:
frog_species_viewer = FrogSpeciesViewer(frog_final_multi)